In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path
import umap 
import plotly.express as px

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

song_embeddings = pd.read_parquet(
    PROCESSED_DIR / "song_embeddings.parquet"
)

song_embeddings.shape

(381, 388)

In [3]:
embedding_cols = [
    col for col in song_embeddings.columns
    if col.startswith("embedding_")
]

X = song_embeddings[embedding_cols].values

X.shape

(381, 384)

In [4]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2, 
    metric="cosine",
    random_state=42
)

coords = reducer.fit_transform(X)

coords.shape

/home/seans/becode/projects/BTS-Song-Atlas-/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(381, 2)

In [5]:
song_atlas = song_embeddings[[
    "track_id", 
    "track_name",
    "album_name",
    "release_year"
]].copy()

song_atlas["x"] = coords[:, 0]
song_atlas["y"] = coords[:, 1]

song_atlas.head()

,track_id,track_name,album_name,release_year,x,y
0,6nt3AoYjkaqXMZhypTBky1,SWIM,KEEP SWIMMING,2026,20.324995,-5.031776
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),KEEP SWIMMING,2026,20.287645,-5.069007
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),KEEP SWIMMING,2026,20.293402,-5.063354
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),KEEP SWIMMING,2026,20.259960,-5.096810
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),KEEP SWIMMING,2026,20.358072,-4.998727


In [6]:
fig = px.scatter(
    song_atlas,
    x="x",
    y="y",
    color="album_name",
    hover_data=["track_name", "album_name", "release_year"],
    title="BTS Song Atlas — Lyric Embedding Map"
)

fig.show()

In [7]:
fig = px.scatter(
    song_atlas, 
    x="x",
    y="y",
    color="album_name",
    hover_name="track_name",
    hover_data=["release_year"],
    width=1200,
    height=800,
    title="BTS Song Atlas"
)

fig.update_traces(marker=dict(size=10))

fig.show()

## Similarity Search

In [8]:
embedding_cols = [
    col for col in song_embeddings.columns
    if col.startswith("embedding_")
]

embedding_matrix = song_embeddings[embedding_cols].values

In [9]:
import re

def canonical_title(title): 
    title = title.lower()
    title = re.sub(r"\s*-\s*live$", "", title)
    title = re.sub(r"\s*-\s*japanese ver\.$", "", title)
    title = re.sub(r"\s*\(.*?\)", "", title)
    title = re.sub(r"[^a-z0-9가-힣\s]", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


In [10]:
def find_similar_songs(song_title, top_n=10, deduplicate=True):
    matches = song_embeddings[
        song_embeddings["track_name"].str.lower() == song_title.lower()
    ]

    if matches.empty:
        print(f"No exact match found for: {song_title}")
        return None

    target_idx = matches.index[0]
    target_vector = embedding_matrix[target_idx].reshape(1, -1)

    similarities = cosine_similarity(target_vector, embedding_matrix)[0]

    results = song_embeddings[[
        "track_id",
        "track_name",
        "album_name",
        "release_year"
    ]].copy()

    results["similarity"] = similarities
    results["canonical_title"] = results["track_name"].apply(canonical_title)

    target_canonical = canonical_title(song_title)

    if deduplicate:
        results = results.sort_values("similarity", ascending=False)
        results = results.drop_duplicates(subset=["canonical_title"], keep="first")

    target_clean = canonical_title(song_title)

    results = results[
        results["canonical_title"] != target_clean
    ]

    results = results[
        ~results["track_name"].str.lower().str.contains(song_title.lower(), regex=False)
    ]

    return results.sort_values("similarity", ascending=False).head(top_n)

In [11]:
find_similar_songs("So What", top_n=10)

,track_id,track_name,album_name,release_year,similarity,canonical_title
192,0G1Y5QG8wULxMzm45vqPIZ,2! 3!,You Never Walk Alone,2017,0.779055,2 3
289,6gGCS5r2t5uxaeBWAPeBp2,Where You From,Skool Luv Affair (Special Addition),2014,0.717444,where you from
252,158kubraiqKrcYpRsDHrBU,Autumn Leaves,The Most Beautiful Moment in Life Pt.2,2015,0.704009,autumn leaves
35,7ldvwszvPNuDAtUM0OJxmj,Stay - Live,PERMISSION TO DANCE ON STAGE - LIVE,2025,0.703323,stay
256,2SKBwYW23iHoB7CrYSeyC9,Hold Me Tight,The Most Beautiful Moment in Life Pt.1,2015,0.676244,hold me tight
276,2uqzfXB7NlBnhQd04W7HaH,Let Me Know,Dark & Wild,2014,0.674675,let me know
350,6wH3AP7b01vpzKYRJhreMy,Black Swan,Black Swan,2020,0.672609,black swan
21,0RTYfrqHvTUxFpmuvSyobG,ON - Live,PERMISSION TO DANCE ON STAGE - LIVE,2025,0.668834,on
108,2QZe2H1f03t5PJWEMg2Mbe,UGH!,MAP OF THE SOUL : 7,2020,0.666488,ugh
165,3zIG0WuI5RoJsFYnErkFDU,Don't Leave Me,FACE YOURSELF,2018,0.659529,dont leave me


In [12]:
find_similar_songs("Dimple", top_n=10)

,track_id,track_name,album_name,release_year,similarity,canonical_title
192,0G1Y5QG8wULxMzm45vqPIZ,2! 3!,You Never Walk Alone,2017,0.649935,2 3
265,6ri9BvvODhRDVWcxL0gIqo,Danger - Japanese Ver.,Wake Up (Standard Edition),2014,0.636296,danger
289,6gGCS5r2t5uxaeBWAPeBp2,Where You From,Skool Luv Affair (Special Addition),2014,0.623167,where you from
252,158kubraiqKrcYpRsDHrBU,Autumn Leaves,The Most Beautiful Moment in Life Pt.2,2015,0.617878,autumn leaves
157,0wqqe8k13bJPxRt7FEmnwS,So What,Love Yourself 轉 'Tear',2018,0.617202,so what
256,2SKBwYW23iHoB7CrYSeyC9,Hold Me Tight,The Most Beautiful Moment in Life Pt.1,2015,0.613852,hold me tight
276,2uqzfXB7NlBnhQd04W7HaH,Let Me Know,Dark & Wild,2014,0.589883,let me know
182,66N38jZOApS3dGhy8ijBVV,Lie,You Never Walk Alone,2017,0.581535,lie
360,0RF2i1nWqMHK0i3xIUAhfl,MIC Drop - Japanese ver.,MIC Drop/DNA/Crystal Snow,2017,0.572761,mic drop
261,0hwev90tfswBoZmgKnJ9F8,Outro: Love Is Not Over,The Most Beautiful Moment in Life Pt.1,2015,0.568191,outro love is not over


In [13]:
find_similar_songs("Boy in Luv", top_n=10)

,track_id,track_name,album_name,release_year,similarity,canonical_title
252,158kubraiqKrcYpRsDHrBU,Autumn Leaves,The Most Beautiful Moment in Life Pt.2,2015,0.693821,autumn leaves
21,0RTYfrqHvTUxFpmuvSyobG,ON - Live,PERMISSION TO DANCE ON STAGE - LIVE,2025,0.679265,on
125,1dS4l6xmdgEhCZTAhdOm4N,Serendipity (Full Length Edition),Love Yourself 結 'Answer',2018,0.678553,serendipity
192,0G1Y5QG8wULxMzm45vqPIZ,2! 3!,You Never Walk Alone,2017,0.673205,2 3
289,6gGCS5r2t5uxaeBWAPeBp2,Where You From,Skool Luv Affair (Special Addition),2014,0.662336,where you from
73,3WMLle4WsAKsZ9G1a5L8c9,Singularity,Proof,2022,0.653960,singularity
157,0wqqe8k13bJPxRt7FEmnwS,So What,Love Yourself 轉 'Tear',2018,0.653023,so what
35,7ldvwszvPNuDAtUM0OJxmj,Stay - Live,PERMISSION TO DANCE ON STAGE - LIVE,2025,0.648598,stay
117,4a9tbd947vo9K8Vti9JwcI,Boy With Luv (feat. Halsey),MAP OF THE SOUL : PERSONA,2019,0.640052,boy with luv
258,1sz4drpDPpHVrPUgnXGaNn,Boyz with Fun,The Most Beautiful Moment in Life Pt.1,2015,0.635998,boyz with fun


In [14]:
find_similar_songs("Black Swan", top_n=10)

,track_id,track_name,album_name,release_year,similarity,canonical_title
157,0wqqe8k13bJPxRt7FEmnwS,So What,Love Yourself 轉 'Tear',2018,0.672609,so what
192,0G1Y5QG8wULxMzm45vqPIZ,2! 3!,You Never Walk Alone,2017,0.666486,2 3
30,4EAIHRqVnaof9FKjYQFDGZ,Boy With Luv (Feat. Halsey) - Live,PERMISSION TO DANCE ON STAGE - LIVE,2025,0.630300,boy with luv
309,4eqHEzFsCCiGALC1MHbqnU,Coffee,"O!RUL8,2?",2013,0.611298,coffee
259,6ZITsss1URNOIKhniylpBP,Converse High,The Most Beautiful Moment in Life Pt.1,2015,0.603801,converse high
35,7ldvwszvPNuDAtUM0OJxmj,Stay - Live,PERMISSION TO DANCE ON STAGE - LIVE,2025,0.593170,stay
120,6Yc3tjFCVR2bfAQFRTZBef,HOME,MAP OF THE SOUL : PERSONA,2019,0.590014,home
165,3zIG0WuI5RoJsFYnErkFDU,Don't Leave Me,FACE YOURSELF,2018,0.589067,dont leave me
274,29SjtKzBoJDgyIBusHylIS,War of Hormone,Dark & Wild,2014,0.583070,war of hormone
57,0U4hNZFJYNFdEAXZcHbkkc,ON,Proof,2022,0.578813,on


In [15]:
# TODO:
# Improve recommendations by combining:
# - lyric embeddings
# - album context
# - release year
# - audio features (Spotify)
# - clustering

## Section 4 — HDBSCAN Semantic clustering

In [16]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=1,
    metric="euclidean",
    prediction_data=True
)

clusters = clusterer.fit_predict(embedding_matrix)

clusters

array([ 2,  2,  2,  2,  2,  2,  2,  2, -1, -1, -1,  9, -1,  2, -1, -1, -1,
       -1, -1, -1, 18, -1,  9, -1,  8, -1, -1, -1,  5, -1, 17, -1,  7, -1,
       -1, 19, 17,  0,  3, -1, -1, 10,  6, -1, -1, -1, -1, 14, 15, 18,  9,
       -1, 10,  8,  5,  0, 17, -1, 11, -1,  7, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, 18, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 11,
       -1, -1, 17, -1, -1,  0,  3,  5, -1, -1, -1, -1, -1, 17, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 17, -1,
       -1, -1, -1, -1, -1, -1, -1,  8, -1, -1, -1, 18,  5, -1, -1, -1, -1,
       18,  0, -1, -1, 16,  3, -1, -1, 13,  8,  5, 13,  0, 18,  5, -1, -1,
       -1, -1,  3, -1, 17, -1, -1, 16, -1,  8, -1, 13, -1, -1, -1, 10, -1,
       -1, -1,  8, 16, -1, -1, 13, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, 17, 10, -1, -1, 19, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, 17, -1, -1, 18,  9, -1, 18, -1, -1,  4, -1,
       15, -1, -1, 15, 14

In [17]:
song_atlas["cluster"] = clusters

song_atlas.head()

,track_id,track_name,album_name,release_year,x,y,cluster
0,6nt3AoYjkaqXMZhypTBky1,SWIM,KEEP SWIMMING,2026,20.324995,-5.031776,2
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),KEEP SWIMMING,2026,20.287645,-5.069007,2
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),KEEP SWIMMING,2026,20.293402,-5.063354,2
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),KEEP SWIMMING,2026,20.259960,-5.096810,2
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),KEEP SWIMMING,2026,20.358072,-4.998727,2


In [18]:
song_atlas["cluster"].value_counts().sort_index()


cluster
-1     245
 0       6
 1       5
 2       9
 3       5
 4       6
 5       9
 6       5
 7       9
 8       7
 9       5
 10      5
 11      5
 12      5
 13      5
 14      6
 15      7
 16      5
 17     12
 18     15
 19      5
Name: count, dtype: int64

In [19]:
n_clusters = len(song_atlas["cluster"].unique()) - (
    -1 in song_atlas["cluster"].unique()
)

print(f"Clusters found: {n_clusters}")

Clusters found: 20


In [20]:
fig = px.scatter(
    song_atlas,
    x="x",
    y="y",
    color="cluster",
    hover_name="track_name",
    hover_data=["album_name", "release_year"],
    width=1200,
    height=800, 
    title="BTS Song Atlas - Semantic Clusters"
)

fig.update_traces(marker=dict(size=10))

fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    xaxis_showticklabels=False,
    yaxis_showticklabels=False
)

fig.show()

In [21]:
for cluster in sorted(song_atlas["cluster"].unique()):

    print("=" * 60)
    print(f"Cluster {cluster}")

display(
    song_atlas[
        song_atlas["cluster"] == cluster
    ][[
        "track_name",
        "album_name",
        "release_year"
    ]].head(20)
)

Cluster -1
Cluster 0
Cluster 1
Cluster 2
Cluster 3
Cluster 4
Cluster 5
Cluster 6
Cluster 7
Cluster 8
Cluster 9
Cluster 10
Cluster 11
Cluster 12
Cluster 13
Cluster 14
Cluster 15
Cluster 16
Cluster 17
Cluster 18
Cluster 19


,track_name,album_name,release_year
35,Stay - Live,PERMISSION TO DANCE ON STAGE - LIVE,2025
196,A Supplementary Story: You Never Walk Alone,You Never Walk Alone,2017
226,Autumn Leaves,The Most Beautiful Moment in Life: Young Forever,2016
252,Autumn Leaves,The Most Beautiful Moment in Life Pt.2,2015
276,Let Me Know,Dark & Wild,2014


In [22]:
song_atlas["cluster"].value_counts().sort_index()

cluster
-1     245
 0       6
 1       5
 2       9
 3       5
 4       6
 5       9
 6       5
 7       9
 8       7
 9       5
 10      5
 11      5
 12      5
 13      5
 14      6
 15      7
 16      5
 17     12
 18     15
 19      5
Name: count, dtype: int64

## Section 5 — K-Means Clustering

In [23]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [26]:
scores = []

for k in range(5, 16): 
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    labels = model.fit_predict(embedding_matrix)

    score = silhouette_score(embedding_matrix, labels)

    scores.append((k, score))

pd.DataFrame(scores, columns=["k", "silhouette_score"])


,k,silhouette_score
0,5,0.054214
1,6,0.056235
2,7,0.058409
3,8,0.066278
4,9,0.076754
5,10,0.084081
6,11,0.082658
7,12,0.084618
8,13,0.093417
9,14,0.098893


In [31]:
song_master = pd.read_csv(PROCESSED_DIR / "song_master.csv")

atlas_export = song_atlas.merge(
    song_master[[
        "track_id", 
        "spotify_url",
        "image_url",
        "lyrics_clean",
        "word_count",
        "character_count"
    ]], 
    on="track_id",
    how="left"
)

# Full metadata export
atlas_export.to_csv(
    PROCESSED_DIR / "song_atlas_full.csv",
    index=False
)

# App map export: only songs with embeddings/coordinates/lyrics
atlas_app = atlas_export[
    atlas_export["lyrics_clean"].notna()
].copy()

atlas_app.to_csv(
    PROCESSED_DIR / "song_atlas.csv",
    index=False
)

print("Full:", atlas_export.shape)
print("App:", atlas_app.shape)

Full: (381, 12)
App: (381, 12)


In [32]:
check = pd.read_csv(PROCESSED_DIR / "song_atlas.csv")

print(check.shape)
print(check.isna().sum().sort_values(ascending=False))
check.head()

(381, 12)
track_id           0
track_name         0
album_name         0
release_year       0
x                  0
y                  0
cluster            0
spotify_url        0
image_url          0
lyrics_clean       0
word_count         0
character_count    0
dtype: int64


,track_id,track_name,album_name,release_year,x,y,cluster,spotify_url,image_url,lyrics_clean,word_count,character_count
0,6nt3AoYjkaqXMZhypTBky1,SWIM,KEEP SWIMMING,2026,20.324995,-5.031776,2,https://open.spotify.com/track/6nt3AoYjkaqXMZh...,https://i.scdn.co/image/ab67616d0000b273dd898e...,"Swim, swim\nWater falling off your skin\nSwim,...",370.0,1872.0
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),KEEP SWIMMING,2026,20.287645,-5.069007,2,https://open.spotify.com/track/7EytKcb3klVPpN5...,https://i.scdn.co/image/ab67616d0000b273dd898e...,"Swim, swim\nWater fallin' off your skin\nSwim,...",362.0,1816.0
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),KEEP SWIMMING,2026,20.293402,-5.063354,2,https://open.spotify.com/track/5dZLsPskKzph16L...,https://i.scdn.co/image/ab67616d0000b273dd898e...,"Swim, swim\nWater falling off your skin\nSwim,...",364.0,1841.0
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),KEEP SWIMMING,2026,20.259960,-5.096810,2,https://open.spotify.com/track/5AL5OrvyIMPqKjl...,https://i.scdn.co/image/ab67616d0000b273dd898e...,"Swim, swim\nWater falling off your skin\nSwim,...",455.0,2312.0
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),KEEP SWIMMING,2026,20.358072,-4.998727,2,https://open.spotify.com/track/3MCJY7lXCHa0UNI...,https://i.scdn.co/image/ab67616d0000b273dd898e...,"Swim, swim\nWater falling off your skin\nSwim,...",370.0,1877.0
